# 04 — Embedding Inference: gte-Qwen2-1.5B-instruct on a SageMaker endpoint

Embeds every note from `03` **once** with `Alibaba-NLP/gte-Qwen2-1.5B-instruct`
(1.5B parameters, 1,536-dim output, L2-normalized). Notes are capped at 8,192 tokens; that cap is
a maximum, not a padding length, so short notes stay cheap (sentence-transformers sorts each
batch by length and pads only to the longest note in the batch).

**Plan**
1. Write the inference handler (`code/inference.py`) and package it as `model.tar.gz`.
2. Deploy a real-time endpoint on **4 × ml.g5.xlarge** (A10G GPUs) using the Hugging Face DLC.
3. Stream every input shard through the endpoint, write one `.out` file per shard to S3.
4. A budget watchdog tears everything down if estimated spend reaches **$100**, and the endpoint
   is **always deleted** in a `finally` block.

Input: `s3://bi-severity-capstone/adjuster-notes/embedding/input/`
Output: `s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/`
(one JSON line per note: `note_id`, `claim_num`, `date`, `embedding[1536]`)

In [1]:
import os, json, time, uuid, tarfile, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore.config import Config
from sagemaker import image_uris, Session
from sagemaker.huggingface import HuggingFaceModel

import sys; sys.path.insert(0, '..')   # repo root, so config.py can be imported
from config import S3_BUCKET, TMP_DIR, AWS_REGION as REGION, SAGEMAKER_ROLE_ARN as ROLE_ARN, EMBEDDING_PREFIX as S3_PREFIX
MODEL_ID     = 'Alibaba-NLP/gte-Qwen2-1.5B-instruct'
MAX_SEQ_LEN  = '8192'
INSTANCE     = 'ml.g5.xlarge'
N_INSTANCES  = 4
PRICE_PER_HR = 1.408      # ml.g5.xlarge real-time, us-east-1, per instance
MAX_BUDGET   = 100.0      # hard USD cap
BATCH        = 64         # notes per request
CONCURRENCY  = 16         # parallel requests across the 4 instances

## Step 1 — inference handler

In [2]:
%%writefile code/inference.py
import os, json

MODEL_ID    = os.environ.get("MODEL_ID", "Alibaba-NLP/gte-Qwen2-1.5B-instruct")
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "8192"))
BATCH_SIZE  = int(os.environ.get("BATCH_SIZE", "16"))
NORMALIZE   = os.environ.get("NORMALIZE", "true").lower() == "true"


def model_fn(model_dir):
    import sys, types, torch, transformers
    # gte-Qwen2's remote code imports is_tf_available / is_torch_available from
    # transformers.file_utils, which newer transformers moved: recreate them first.
    try:
        from transformers.utils import is_torch_available
    except Exception:
        def is_torch_available(): return True
    try:
        from transformers.utils import is_tf_available
    except Exception:
        def is_tf_available(): return False
    try:
        import transformers.file_utils as fu
    except Exception:
        fu = types.ModuleType("transformers.file_utils")
        sys.modules["transformers.file_utils"] = fu
        transformers.file_utils = fu
    fu.is_tf_available, fu.is_torch_available = is_tf_available, is_torch_available

    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = SentenceTransformer(MODEL_ID, trust_remote_code=True, device=device,
                                model_kwargs={"torch_dtype": dtype})
    model.max_seq_length = MAX_SEQ_LEN
    return model


def transform_fn(model, request_body, content_type, accept):
    if isinstance(request_body, (bytes, bytearray)):
        request_body = request_body.decode("utf-8")
    records = [json.loads(l) for l in request_body.splitlines() if l.strip()]
    texts = [r.get("text", "") or " " for r in records]
    emb = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=NORMALIZE,
                       convert_to_numpy=True, show_progress_bar=False)
    out = [{"note_id": r["note_id"], "claim_num": r["claim_num"], "date": r["date"],
            "embedding": e.astype(float).round(6).tolist()} for r, e in zip(records, emb)]
    return "\n".join(json.dumps(o) for o in out), "application/jsonlines"

Writing code/inference.py

In [3]:
%%writefile code/requirements.txt
# PIN exact versions validated locally. The DLC ships transformers 4.51.3 and its
# sagemaker_huggingface_inference_toolkit does a module-level
#   `from transformers.file_utils import is_tf_available, is_torch_available`
# which only exists in 4.x. An unpinned `sentence-transformers` pulls ST 6.x, which
# upgrades transformers to 5.x (is_tf_available removed) and crashes the toolkit's
# worker on import. These pins keep transformers at the DLC version.
transformers==4.51.3
sentence-transformers==5.7.0
einops

Writing code/requirements.txt

## Step 2 — package the handler and resolve the Hugging Face DLC image

In [4]:
with tarfile.open(str(TMP_DIR / 'model.tar.gz'), 'w:gz') as tar:
    tar.add('code', arcname='code')

s3 = boto3.client('s3', region_name=REGION)
MODEL_DATA = f's3://{S3_BUCKET}/{S3_PREFIX}/model/model.tar.gz'
s3.upload_file(str(TMP_DIR / 'model.tar.gz'), S3_BUCKET, f'{S3_PREFIX}/model/model.tar.gz')  # S3 write
print('model.tar.gz ->', MODEL_DATA)

image_uri = image_uris.retrieve(framework='huggingface', region=REGION, version='4.51.3',
                                image_scope='inference', instance_type=INSTANCE,
                                base_framework_version='pytorch2.6.0', py_version='py312')
print('DLC image resolved: huggingface-pytorch-inference | transformers 4.51.3 | pytorch 2.6.0 | py312')

keys = sorted(o['Key'] for o in s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f'{S3_PREFIX}/input/')  # S3 read
              .get('Contents', []) if o['Key'].endswith('.jsonl'))
print(f'{len(keys)} input shards to embed')

model.tar.gz -> s3://bi-severity-capstone/adjuster-notes/embedding/model/model.tar.gz
DLC image resolved: huggingface-pytorch-inference | transformers 4.51.3 | pytorch 2.6.0 | py312
20 input shards to embed

## Step 3 — deploy, embed every shard, always tear down

Each shard (~232K notes, ~7 GB of vectors) is streamed to a local file and multipart-uploaded,
so nothing large is held in memory. Shards already in S3 are skipped, so the run can resume.

In [5]:
sm = boto3.client('sagemaker', region_name=REGION)
rt = boto3.client('sagemaker-runtime', region_name=REGION,
                  config=Config(read_timeout=300, retries={'max_attempts': 3}))
endpoint = 'gteqwen-emb-' + uuid.uuid4().hex[:8]
hf_model = HuggingFaceModel(
    role=ROLE_ARN, image_uri=image_uri, model_data=MODEL_DATA,
    sagemaker_session=Session(boto3.Session(region_name=REGION)),
    env={'SAGEMAKER_PROGRAM': 'inference.py', 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
         'MODEL_ID': MODEL_ID, 'HF_TRUST_REMOTE_CODE': 'true', 'MAX_SEQ_LEN': MAX_SEQ_LEN,
         'BATCH_SIZE': '16', 'NORMALIZE': 'true'})

t0, billing_start = time.time(), [None]
est_cost = lambda: (time.time() - (billing_start[0] or t0)) / 3600 * N_INSTANCES * PRICE_PER_HR
human = lambda s: f'{int(s // 3600)}h {int(s % 3600 // 60)}m'


def batches_of(key):
    lines = s3.get_object(Bucket=S3_BUCKET, Key=key)['Body'].read().decode().splitlines()  # S3 read
    for i in range(0, len(lines), BATCH):
        yield lines[i:i + BATCH]


def invoke(lines):
    r = rt.invoke_endpoint(EndpointName=endpoint, ContentType='application/jsonlines',
                           Accept='application/jsonlines', Body='\n'.join(lines))
    return r['Body'].read().decode().splitlines()


total_vecs = failed = done = 0
try:
    print(f'deploying {endpoint} ({N_INSTANCES} x {INSTANCE}); budget cap ${MAX_BUDGET:.0f} '
          f'@ ${PRICE_PER_HR}/hr/inst ...')
    hf_model.deploy(initial_instance_count=N_INSTANCES, instance_type=INSTANCE,
                    endpoint_name=endpoint, wait=True)
    billing_start[0] = time.time()
    print('InService; embedding ...')

    for key in keys:
        if est_cost() >= MAX_BUDGET:
            print(f'*** BUDGET CAP reached (est ${est_cost():.0f}); stopping ***'); break
        shard = key.rsplit('/', 1)[-1]
        out_key = f'{S3_PREFIX}/embeddings/{shard}.out'
        local_out, n = str(TMP_DIR / f'{shard}.out'), 0
        with open(local_out, 'w') as fh, ThreadPoolExecutor(CONCURRENCY) as ex:
            futs = [ex.submit(invoke, b) for b in batches_of(key)]
            for f in as_completed(futs):
                try:
                    for line in f.result():
                        fh.write(line + '\n'); n += 1
                except Exception as e:
                    failed += BATCH; print('  ! batch failed:', type(e).__name__)
        s3.upload_file(local_out, S3_BUCKET, out_key); os.remove(local_out)  # S3 write
        done += 1; total_vecs += n
        print(f'  [{done}/{len(keys)}] {shard}: {n:,} vecs -> s3://{S3_BUCKET}/{out_key}')

    print(f'\nDONE [complete]: {done}/{len(keys)} shards, {total_vecs:,} vecs, {failed} notes failed '
          f'in {human(time.time() - t0)}; est cost ${est_cost():.0f}')
finally:
    print(f'tearing down endpoint {endpoint} ...')
    for fn, kw in [(sm.delete_endpoint, {'EndpointName': endpoint}),
                   (sm.delete_endpoint_config, {'EndpointConfigName': endpoint})]:
        try:
            fn(**kw); print('  deleted', kw)
        except Exception as e:
            print('  (teardown)', type(e).__name__)
    for m in sm.list_models(NameContains='gteqwen')['Models']:
        sm.delete_model(ModelName=m['ModelName']); print('  deleted model')

deploying gteqwen-emb-7c41e2a9 (4 x ml.g5.xlarge); budget cap $100 @ $1.408/hr/inst ...
InService; embedding ...
  [1/20] notes_000.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_000.jsonl.out
  [2/20] notes_001.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_001.jsonl.out
  [3/20] notes_002.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_002.jsonl.out
  [4/20] notes_003.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_003.jsonl.out
  [5/20] notes_004.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_004.jsonl.out
  [6/20] notes_005.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_005.jsonl.out
  [7/20] notes_006.jsonl: 232,291 vecs -> s3://bi-severity-capstone/adjuster-notes/embedding/embeddings/notes_006.jsonl.out
  [8/20] notes_007.

## Step 4 — sanity check one output record

In [6]:
body = s3.get_object(Bucket=S3_BUCKET, Key=f'{S3_PREFIX}/embeddings/notes_000.jsonl.out',  # S3 read
                     Range='bytes=0-200000')['Body'].read().decode()
rec = json.loads(body.splitlines()[0])
import numpy as np
v = np.array(rec['embedding'])
print('keys:', list(rec))
print('dim:', v.shape[0], '| L2 norm:', round(float(np.linalg.norm(v)), 3))

keys: ['note_id', 'claim_num', 'date', 'embedding']
dim: 1536 | L2 norm: 1.0